In [1]:
# ============================================
# CELL 1 — Install & Import Dependencies
# ============================================
!pip install requests pandas tqdm --quiet

import requests
import json
import time
from tqdm import tqdm


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================
# CELL 2 — Konfigurasi API
# ============================================
# Daftar API key gratis di https://pokemontcg.io/profile (opsional tapi disarankan)
# Tanpa key: limit 1,000 request/hari
# Dengan key: limit 20,000 request/hari
API_KEY = ""  # isi kalau sudah punya, atau biarkan kosong untuk testing

BASE_URL = "https://api.pokemontcg.io/v2/cards"
HEADERS = {"X-Api-Key": API_KEY} if API_KEY else {}

In [3]:
# ============================================
# CELL 3 — Fungsi Ambil Data per Set
# ============================================
def fetch_cards_by_set(set_id, page_size=250):
    """
    Ambil semua kartu dari satu set tertentu.
    Contoh set_id: 'base1' (Base Set), 'base2' (Jungle), dll.
    """
    all_cards = []
    page = 1

    while True:
        params = {
            "q": f"set.id:{set_id}",
            "page": page,
            "pageSize": page_size
        }
        response = requests.get(BASE_URL, headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code} — {response.text}")
            break

        data = response.json()
        cards = data.get("data", [])

        if not cards:
            break

        all_cards.extend(cards)
        page += 1
        time.sleep(0.3)  # sopan ke API, hindari rate limit

    return all_cards

In [4]:
# ============================================
# CELL 4 — Fungsi Ekstrak Field yang Relevan
# ============================================
def extract_relevant_fields(card):
    """
    Ambil hanya field yang sesuai dengan rancangan PokeScan:
    - Identitas kartu (untuk Stage 2: Card Identification)
    - Gambar referensi (untuk Stage 2: Feature Extraction / embedding)
    - Harga TCGPlayer & Cardmarket (untuk Stage 4: Price Lookup)
    """
    tcgplayer = card.get("tcgplayer", {})
    tcg_prices = tcgplayer.get("prices", {})

    cardmarket = card.get("cardmarket", {})
    cm_prices = cardmarket.get("prices", {})

    # Ambil varian harga TCGPlayer (holofoil, normal, reverseHolofoil, dll)
    tcg_variant = {}
    for variant_name, variant_data in tcg_prices.items():
        tcg_variant[variant_name] = {
            "low": variant_data.get("low"),
            "mid": variant_data.get("mid"),
            "high": variant_data.get("high"),
            "market": variant_data.get("market"),
        }

    extracted = {
        "card_id": card.get("id"),
        "name": card.get("name"),
        "number": card.get("number"),
        "rarity": card.get("rarity"),
        "supertype": card.get("supertype"),          # Pokémon / Trainer / Energy
        "subtypes": card.get("subtypes", []),          # e.g. ["Basic", "VMAX"]
        "set": {
            "id": card.get("set", {}).get("id"),
            "name": card.get("set", {}).get("name"),
            "series": card.get("set", {}).get("series"),
            "release_date": card.get("set", {}).get("releaseDate"),
            "total_cards": card.get("set", {}).get("printedTotal"),
        },
        "images": {
            "small": card.get("images", {}).get("small"),
            "large": card.get("images", {}).get("large"),
        },
        "prices": {
            "tcgplayer_url": tcgplayer.get("url"),
            "tcgplayer_updated": tcgplayer.get("updatedAt"),
            "tcgplayer_variants": tcg_variant,
            "cardmarket_url": cardmarket.get("url"),
            "cardmarket_updated": cardmarket.get("updatedAt"),
            "cardmarket_avg_sell": cm_prices.get("averageSellPrice"),
            "cardmarket_trend": cm_prices.get("trendPrice"),
            "cardmarket_avg1": cm_prices.get("avg1"),
            "cardmarket_avg7": cm_prices.get("avg7"),
            "cardmarket_avg30": cm_prices.get("avg30"),
        }
    }
    return extracted

In [5]:
# ============================================
# CELL 5 — Jalankan untuk Beberapa Set Populer
# ============================================
# Mulai dari set-set populer dulu untuk testing (scope kecil sesuai rekomendasi)
target_sets = [
    "base1",   # Base Set — termasuk Charizard iconic
    "base2",   # Jungle
    "base3",   # Fossil
]

all_extracted_cards = []

for set_id in tqdm(target_sets, desc="Fetching sets"):
    print(f"\nMengambil data set: {set_id}")
    raw_cards = fetch_cards_by_set(set_id)
    print(f"  → Ditemukan {len(raw_cards)} kartu")

    for card in raw_cards:
        extracted = extract_relevant_fields(card)
        all_extracted_cards.append(extracted)

print(f"\nTotal kartu terkumpul: {len(all_extracted_cards)}")

Fetching sets:   0%|          | 0/3 [00:00<?, ?it/s]


Mengambil data set: base1


Fetching sets:  33%|███▎      | 1/3 [00:01<00:02,  1.34s/it]

Error: 500 — 
  → Ditemukan 0 kartu

Mengambil data set: base2


Fetching sets:  67%|██████▋   | 2/3 [00:02<00:01,  1.24s/it]

Error: 500 — 
  → Ditemukan 0 kartu

Mengambil data set: base3


Fetching sets: 100%|██████████| 3/3 [00:03<00:00,  1.07s/it]

Error: 500 — 
  → Ditemukan 0 kartu

Total kartu terkumpul: 0


In [6]:
# ============================================
# CELL 6 — Simpan ke JSON
# ============================================
output_filename = "pokemon_cards_dataset.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(all_extracted_cards, f, indent=2, ensure_ascii=False)

print(f"Dataset tersimpan di: {output_filename}")
print(f"Total record: {len(all_extracted_cards)}")

Dataset tersimpan di: pokemon_cards_dataset.json
Total record: 0


In [7]:
# ============================================
# CELL 7 — Preview Hasil (Cek Sample Data)
# ============================================
import pandas as pd

# Tampilkan sample pertama dengan format rapi
print(json.dumps(all_extracted_cards[0], indent=2))

# Ringkasan dalam bentuk tabel
df_preview = pd.DataFrame([{
    "name": c["name"],
    "set": c["set"]["name"],
    "rarity": c["rarity"],
    "tcgplayer_variants": list(c["prices"]["tcgplayer_variants"].keys()),
    "cardmarket_avg": c["prices"]["cardmarket_avg_sell"],
    "has_image": bool(c["images"]["large"])
} for c in all_extracted_cards])

df_preview.head(10)

IndexError: list index out of range

In [ ]:
# ============================================
# CELL 8 (Opsional) — Download Gambar Kartu
# ============================================
# Untuk keperluan training embedding di Stage 2 (EfficientNet)
import os
import time
import requests
from tqdm import tqdm

# Mengubah target direktori menjadi "pokemon-cards"
os.makedirs("pokemon-cards", exist_ok=True)

def download_card_image(card, save_dir="pokemon-cards"):
    img_url = card["images"]["large"]
    if not img_url:
        return None

    card_id = card["card_id"]
    save_path = os.path.join(save_dir, f"{card_id}.png")

    if os.path.exists(save_path):  # skip kalau sudah ada
        return save_path

    try:
        resp = requests.get(img_url, timeout=10)
        if resp.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(resp.content)
            return save_path
    except Exception as e:
        print(f"Gagal download {card_id}: {e}")
    return None

# Download semua gambar (jalankan hati-hati, ada delay biar tidak overload server)
for card in tqdm(all_extracted_cards, desc="Downloading images"):
    download_card_image(card)
    time.sleep(0.1)

print("Selesai download gambar ke folder 'pokemon-cards/'")